In [1]:
!tree ../dataset

../dataset
├── en_espanol
│   ├── docx2txt.py
│   ├── Второй_жанр_исходная.txt
│   └── Первый_жанр_исходная.txt
├── Второй_жанр_исходная.txt
├── Первый_жанр_исходная.txt
├── Сокращение по частям речи
│   ├── 1.Первый жанр исходная выборка.txt
│   ├── 2.Первый жанр без клауз, включающих наречия.txt
│   ├── 3.Первый жанр без клауз, включающих глаголы.txt
│   ├── 4.Первый жанр без клауз, включающих глаголы и наречия.txt
│   ├── 5.без клауз, включающих местоимения.txt
│   ├── 6.без слов функциональных.txt
│   ├── Без прилагательных второй жанр.txt
│   ├── Без прилагательных первый жанр.txt
│   ├── Второй_жанр без клауз, включающих местоимения.txt
│   ├── Второй_жанр без слов функциональных.txt
│   └── Случайные выборки.txt
└── сокращение по частотности
    ├── 1а_ без сокращений.txt
    ├── 1б_Изъяты лексемы с частотой выше 100.txt
    ├── 1в_Изъяты лексемы с частотой выше 49.txt
    ├── 1г_Изъяты лексемы с частотой выше 29.txt
    ├── 1д_Изъяты лексемы с частотой выше 9.txt
    ├── 1е_Изъ

In [2]:
import json

def save_results(results, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

In [3]:
import funciones
from utils import train_wrapper
import warnings
import os
import torch
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
from contextlib import redirect_stderr
import multiprocessing as mp
import numpy as np
#import wandb
import nbformat
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sklearn.metrics import classification_report, confusion_matrix
import gc
import re
import nltk
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from transformers import BertForSequenceClassification, AdamW
from razdel import sentenize
import numpy as np

# Suprimir warnings específicos
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
mp.set_start_method('spawn', force=True)


import json


def convert_numpy(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.float32, np.float64, np.int32, np.int64)):
        return obj.item()
    return obj

def save_results(results, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=4, ensure_ascii=False, default=convert_numpy)

        
def create_custom_config(model_name, model_type, dataset):
    """Crea configuración de entrenamiento adaptativa"""
    common_config = {
#         'dataset_name':dataset.get('name', ''),
        'model_name': model_name,
        'model_type': model_type,
        'num_repeats': 6,
        'test_size': 0.2,
        'threshold': 0.5
    }

    # Configuraciones basadas en frecuencia
    freq_threshold = dataset.get('freq_threshold')
    if freq_threshold in [100, 49, 29, 9, 5, 3]:
        configs = {
            100: (52, 16, 6, 2e-5),
            49: (60, 16, 6, 3e-5),
            29: (51, 16, 6, 3e-5),
            9: (45, 16, 6, 4e-5),
            5: (150, 16, 6, 5e-5),
            3: (150, 16, 6, 5e-5)
        }
        max_len, batch, epochs, lr = configs[freq_threshold]
        return funciones.TrainingConfig(
            **common_config,
            max_length=max_len,
            batch_size=batch,
            epochs=epochs,
            learning_rate=lr
        )

    # Configuración por nombre de dataset
    dataset_name = dataset.get('name', '')
    for x in ['1', '2', '3', '4', '5', '6']:
        if x in dataset_name:
            configs = {
                '1': (60, 16, 6, 5e-5),
                '2': (60, 16, 6, 5e-5),
                '3': (60, 16, 6, 5e-5),
                '4': (60, 16, 6, 5e-5),
                '5': (60, 16, 6, 5e-5),
                '6': (60, 16, 6, 5e-5)
            }
            max_len, batch, epochs, lr = configs[x]
            return funciones.TrainingConfig(
                **common_config,
                max_length=max_len,
                batch_size=batch,
                epochs=epochs,
                learning_rate=lr
            )


    # Configuración por defecto
    return funciones.TrainingConfig(
        **common_config,
        max_length=60,
        batch_size=32,
        epochs=4,
        learning_rate=3e-5
    )

models = [
    {'model':'DeepPavlov/rubert-base-cased',
     'name':'DeepPavlov-rubert-base',
     'type': 'bert'}, # Modelo original (ruso)
    {'model':'bert-base-multilingual-cased',
     'name':'BERT multilingual',
     'type': 'bert'}, # BERT multilingüe
    {'model':'distilbert-base-multilingual-cased',
     'name':'distilbert-base-multilingual',
     'type': 'bert'}, # Versión ligera de BERT multilingüe
    {'model':'roberta-base',
     'name':'roberta-base',
     'type': 'bert'}, # RoBERTa (principalmente inglés)

    {'model':'xlnet-base-cased',
     'name':'XLNet Base',
     'type': 'bert'}, # XLNet (multilingüe efectivo)
    {'model':'xlm-roberta-base',
     'name':'XLM-RoBERTa Base',
     'type': 'bert'}, # XLM-RoBERTa (multilingüe)


    {'model': 'gpt2',
     'name': 'gpt2',
     'type': 'gpt'},  # GPT model
    {'model': 'facebook/opt-125m',
     'name': 'facebook/opt-125m',
     'type': 'gpt'},  # GPT model
    {'model': 'sberbank-ai/rugpt3small_based_on_gpt2',
     'name': 'rugpt3small',
     'type': 'gpt'}, # GPT model (ruso)
#         {
#         "model": "facebook/mbart-large-50",
#         "name": "mBART Large 50",
#         "type": "gpt"
#         }
    ]

datasets = [
    #deberia adicionar el set original?
        {
            'path1': '../dataset/сокращение по частотности/1б_Изъяты лексемы с частотой выше 100.txt',
            'path2': '../dataset/сокращение по частотности/2б_Изъяты лексемы с частотой выше 100.txt',
            'name': 'Изъяты лексемы с частотой выше 100',
            'type': 'freq',
            'freq': 100
        },
        {
            'path1': '../dataset/сокращение по частотности/1в_Изъяты лексемы с частотой выше 49.txt',
            'path2': '../dataset/сокращение по частотности/2в_Изъяты лексемы с частотой выше 49.txt',
            'name': 'Изъяты лексемы с частотой выше 49',
            'type': 'freq',
            'freq': 49
        },
        {
            'path1': '../dataset/сокращение по частотности/1г_Изъяты лексемы с частотой выше 29.txt',
            'path2': '../dataset/сокращение по частотности/2г_Изъяты лексемы с частотой выше 29.txt',
            'name': 'Изъяты лексемы с частотой выше 29',
            'type': 'freq',
            'freq': 29
        },
        {
            'path1': '../dataset/сокращение по частотности/1д_Изъяты лексемы с частотой выше 9.txt',
            'path2': '../dataset/сокращение по частотности/2д_Изъяты лексемы с частотой выше 9.txt',
            'name': 'Изъяты лексемы с частотой выше 9',
            'type': 'freq',
            'freq': 9
        },
        {
            'path1': '../dataset/сокращение по частотности/1е_Изъяты лексемы с частотой выше 5.txt',
            'path2': '../dataset/сокращение по частотности/2е_Изъяты лексемы с частотой выше 5.txt',
            'name': 'Изъяты лексемы с частотой выше 5',
            'type': 'freq',
            'freq': 5
        },
        {
            'path1': '../dataset/сокращение по частотности/1ё_Изъяты лексемы с частотой выше 3.txt',
            'path2': '../dataset/сокращение по частотности/2ё_Изъяты лексемы с частотой выше 3.txt',
            'name': 'Изъяты лексемы с частотой выше 3',
            'type': 'freq',
            'freq': 3
        },

            {
            'path1': '../dataset/Сокращение по частям речи/Без прилагательных первый жанр.txt',
            'path2': '../dataset/Сокращение по частям речи/Без прилагательных второй жанр.txt',
            'name': 'Без прилагательных первый-второй жанр',#litle correction
            'type': 'pos',
            'freq': None
            },
          {
            'path1': '../dataset/Сокращение по частям речи/1.Первый жанр исходная выборка.txt',
            'path2': '../dataset/Второй_жанр_исходная.txt',
            'name': '1.Первый жанр исходная выборка',# este el el orignal
            'type': 'pos',
            'freq': None

        },
        {
            'path1': '../dataset/Сокращение по частям речи/2.Первый жанр без клауз, включающих наречия.txt',
            'path2': '../dataset/Второй_жанр_исходная.txt',
            'name': '2.Первый жанр без клауз, включающих наречия',
            'type': 'pos',
            'freq': None
        },
        {
            'path1': '../dataset/Сокращение по частям речи/3.Первый жанр без клауз, включающих глаголы.txt',
            'path2': '../dataset/Второй_жанр_исходная.txt',
            'name': '3.Первый жанр без клауз, включающих глаголы',
            'type': 'pos',
            'freq': None
        },
        {
            'path1': '../dataset/Сокращение по частям речи/4.Первый жанр без клауз, включающих глаголы и наречия.txt',
            'path2': '../dataset/Второй_жанр_исходная.txt',
            'name': '4.Первый жанр без клауз, включающих глаголы и наречия',
            'type': 'pos',
            'freq': None
        },
        {
            'path1': '../dataset/Сокращение по частям речи/5.без клауз, включающих местоимения.txt',
            'path2': '../dataset/Второй_жанр_исходная.txt',
            'name': '5.без клауз, включающих местоимения.txt',
            'type': 'pos',
            'freq': None
        },
        {
            'path1': '../dataset/Сокращение по частям речи/6.без слов функциональных.txt',
            'path2': '../dataset/Второй_жанр_исходная.txt',
            'name': '6.без слов функциональных.txt',
            'type': 'pos',
            'freq': None
        },
    
    ]



russian_templates = [
    "Этот литературный фрагмент принадлежит к жанру {}.",
    "Данный текст является примером жанра {}.",
    "Стилистические особенности указывают на жанр {}.",
    "Жанровая принадлежность этого текста - {}.",
    "Эксперты классифицируют этот текст как жанр {}.",
    "Это типичный пример жанра {} в русской литературе.",
    "По ключевым характеристикам это текст жанра {}.",
    "Данное произведение относится к направлению {}.",
    "Анализ содержания позволяет отнести текст к жанру {}.",
    "Этот отрывок характерен для жанра {}."
]


def main():

    results = []
    for model in models:
        for dataset in datasets:
            config = create_custom_config(model['model'], model['type'], dataset)
            
            
            # Entrenar y limpiar memoria
            torch.cuda.empty_cache()
            result = funciones.train_and_evaluate_dataset(
                dataset['path1'],
                dataset['path2'],
                config,
                dataset['name'],
                dataset['type']
                
            )
            results.append(result) 
#             if model['type'] = 'gpt': #aplicar zero shot
                
        
        del model
        torch.cuda.empty_cache()
        gc.collect()
        
    
    save_results(results, 'resultados_Step_8_test_2.json') 
    
    
    

    
if __name__ == "__main__":
    main()

2025-05-11 17:42:22.566572: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


train_Изъяты лексемы с частотой выше 100.csv
test_Изъяты лексемы с частотой выше 100.csv
Train labels sample: tensor([0, 0, 0, 0, 1]), Shape: torch.Size([864])
Test labels sample: tensor([0, 0, 0, 0, 0]), Shape: torch.Size([195])
##########################DeepPavlov/rubert-base-cased---bert####################


KeyboardInterrupt: 

In [ ]:
#batch_size=128

In [ ]:
#display_results()

In [ ]:
import os
import IPython

# Reiniciar el kernel
IPython.display.display(IPython.display.Javascript("Jupyter.notebook.kernel.restart()"))

# Apagar el kernel después de reiniciar
os._exit(0)

In [ ]:
## hacer zero shot para todos los  modelos


In [ ]:
import os
import re
import json
import gc
import warnings
import numpy as np
import torch
from tqdm import tqdm
from razdel import sentenize
from typing import List, Tuple, Dict, Any, Optional
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, 
                            average_precision_score, log_loss, confusion_matrix)

# Configuración de warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

class Config:
    """Clase de configuración para el experimento zero-shot."""
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.max_length = 256
        self.min_length_threshold = 6
        self.num_repeats = 1
        self.model_type = 'gpt'
        self.genres = ("определение", "описание")

class DataLoader:
    """Cargador de datos mejorado con manejo de errores y validación."""
    def __init__(self, file_paths: Tuple[str, str]):
        self.file_paths = file_paths
        self.validate_paths()
    
    def validate_paths(self):
        """Valida que los archivos existan."""
        for path in self.file_paths:
            if not os.path.exists(path):
                raise FileNotFoundError(f"Archivo no encontrado: {path}")
    
    def load_file(self, path: str, label: int) -> List[Tuple[str, int]]:
        """Carga un archivo y devuelve una lista de (texto, etiqueta)."""
        with open(path, 'r', encoding='utf-8') as f:
            return [(line.strip(), label) for line in f if line.strip()]
    
    def __call__(self) -> Dict[str, List[Tuple[str, int]]]:
        """Carga todos los datos."""
        data1 = self.load_file(self.file_paths[0], 0)
        data2 = self.load_file(self.file_paths[1], 1)
        return {'original_data': data1 + data2}

class TextProcessor:
    """Procesador de texto con limpieza y división en oraciones."""
    def __init__(self, tokenizer, min_length: int = 6):
        self.tokenizer = tokenizer
        self.min_length = min_length
    
    def clean_text(self, text: str) -> str:
        """Limpia el texto para una mejor división en oraciones."""
        # Manejo de casos especiales de puntuación
        replacements = [
            (r'\.,', '. TEMP_MARKER ,'),
            (r'\.;', '. TEMP_MARKER ;'),
            (r'\. ([a-zа-я])', r'. TEMP_MARKER \1'),
            (r'(\w)([А-Я])', r'\1. \2'),
            (r'\.(\s*\d)', r'\1')  # Eliminar puntos antes de números
        ]
        
        for pattern, repl in replacements:
            text = re.sub(pattern, repl, text)
        
        return text
    
    def split_sentences(self, text: str) -> List[str]:
        """Divide el texto en oraciones usando razdel."""
        return [sentence.text for sentence in sentenize(text)]
    
    def process_text(self, text: str) -> List[str]:
        """Procesa un texto completo y devuelve oraciones limpias."""
        cleaned = self.clean_text(text)
        no_brackets = re.sub(r'\[.*?\]', '', cleaned).strip()
        sentences = self.split_sentences(no_brackets)
        return [re.sub(r'\s*TEMP_MARKER', ' ', s).strip() for s in sentences]
    
    def filter_sentences(self, sentences: List[str]) -> List[str]:
        """Filtra oraciones basadas en la longitud de tokens."""
        return [
            s for s in sentences 
            if len(self.tokenizer.encode(s, truncation=False)) >= self.min_length
        ]
    
    def __call__(self, data: List[Tuple[str, int]]) -> List[Tuple[str, int]]:
        """Procesa una lista de textos con etiquetas."""
        processed = []
        for text, label in data:
            sentences = self.process_text(text)
            valid_sentences = self.filter_sentences(sentences)
            processed.extend([(s, label) for s in valid_sentences])
        return processed

class ZeroShotClassifier:
    """Clasificador zero-shot con manejo de modelos generativos."""
    def __init__(self, config: Config):
        self.config = config
        self.model = self.load_model()
    
    def load_model(self) -> AutoModelForCausalLM:
        """Carga el modelo con configuraciones optimizadas."""
        model = AutoModelForCausalLM.from_pretrained(
            self.config.model_name,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        ).to(self.config.device)
        model.eval()
        return model
    
    def create_prompt(self, text: str) -> str:
        """Genera el prompt para clasificación zero-shot."""
        genre1, genre2 = self.config.genres
        return f"""Определите жанр текста (только цифру):
1 - {genre1} (строгое определение понятия)
2 - {genre2} (подробное описание характеристик)

Текст: {text}
Жанр (1 или 2): """
    
    def parse_response(self, response: str) -> int:
        """Interpreta la respuesta del modelo."""
        response = response.strip().lower()
        genre1, genre2 = (g.lower() for g in self.config.genres)
        
        if genre1 in response:
            return 0
        elif genre2 in response:
            return 1
        return -1  # Respuesta no válida
    
    def predict(self, text: str) -> Tuple[int, str]:
        """Realiza una predicción para un texto dado."""
        prompt = self.create_prompt(text)
        inputs = self.config.tokenizer(
            prompt, 
            return_tensors="pt", 
            truncation=True, 
            max_length=self.config.max_length
        ).to(self.config.device)
        
        try:
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=10,
                pad_token_id=self.config.tokenizer.pad_token_id
            )
            response = self.config.tokenizer.decode(
                outputs[0], 
                skip_special_tokens=True
            )
            return self.parse_response(response), response
        except Exception as e:
            print(f"Error en predicción: {e}")
            return -1, ""
    
    def evaluate(self, test_data: List[Tuple[str, int]]) -> Dict[str, Any]:
        """Evalúa el modelo en un conjunto de datos."""
        true_labels = []
        predictions = []
        logits = []
        responses = []
        
        for text, label in tqdm(test_data, desc="Evaluando"):
            pred, response = self.predict(text)
            if pred != -1:
                true_labels.append(label)
                predictions.append(pred)
                logits.append([1.0 - pred, pred])  # Logits simulados
                responses.append(response)
        
        if not true_labels:
            return self.empty_metrics()
        
        return self.calculate_metrics(true_labels, predictions, logits, responses)
    
    @staticmethod
    def empty_metrics() -> Dict[str, Any]:
        """Métricas vacías para casos sin predicciones válidas."""
        return {
            'accuracy': 0.0,
            'f1_scores': {'weighted': 0.0, 'macro': 0.0, 'class_0': 0.0, 'class_1': 0.0},
            'roc_auc': 0.0,
            'pr_auc': 0.0,
            'log_loss': 0.0,
            'confusion_matrix': [[0, 0], [0, 0]],
            'true_labels': [],
            'predictions': [],
            'responses': []
        }
    
    @staticmethod
    def calculate_metrics(true_labels, predictions, logits, responses) -> Dict[str, Any]:
        """Calcula todas las métricas de evaluación."""
        return {
            'accuracy': accuracy_score(true_labels, predictions),
            'f1_scores': {
                'weighted': f1_score(true_labels, predictions, average='weighted'),
                'macro': f1_score(true_labels, predictions, average='macro'),
                'class_0': f1_score(true_labels, predictions, average=None)[0],
                'class_1': f1_score(true_labels, predictions, average=None)[1]
            },
            'roc_auc': roc_auc_score(true_labels, [l[1] for l in logits]),
            'pr_auc': average_precision_score(true_labels, [l[1] for l in logits]),
            'log_loss': log_loss(true_labels, logits),
            'confusion_matrix': confusion_matrix(true_labels, predictions).tolist(),
            'true_labels': true_labels,
            'predictions': predictions,
            'responses': responses
        }

class ExperimentRunner:
    """Ejecuta el experimento completo con múltiples modelos."""
    def __init__(self, data_paths: Tuple[str, str], models: List[Dict[str, str]]):
        self.data_paths = data_paths
        self.models = models
        self.results = []
    
    def run_single_model(self, model_info: Dict[str, str]) -> Dict[str, Any]:
        """Ejecuta el experimento para un solo modelo."""
        print(f"\nEvaluando modelo: {model_info['name']}")
        
        config = Config(model_info['model'])
        data_loader = DataLoader(self.data_paths)
        processor = TextProcessor(config.tokenizer, config.min_length_threshold)
        
        # Cargar y procesar datos
        raw_data = data_loader()['original_data']
        processed_data = processor(raw_data)
        
        # Ejecutar múltiples repeticiones
        metrics = {
            'model_name': model_info['name'],
            'dataset_name': 'Datos originales',
            'repetitions': []
        }
        
        for _ in range(config.num_repeats):
            classifier = ZeroShotClassifier(config)
            result = classifier.evaluate(processed_data)
            metrics['repetitions'].append(result)
            
            # Limpieza de memoria
            del classifier
            torch.cuda.empty_cache()
            gc.collect()
        
        # Calcular promedios
        self.calculate_averages(metrics)
        return metrics
    
    @staticmethod
    def calculate_averages(metrics: Dict[str, Any]):
        """Calcula promedios y desviaciones estándar de las repeticiones."""
        keys = ['accuracy', 'roc_auc', 'pr_auc', 'log_loss']
        f1_keys = ['weighted', 'macro', 'class_0', 'class_1']
        
        # Métricas principales
        for key in keys:
            values = [rep[key] for rep in metrics['repetitions']]
            metrics[f'avg_{key}'] = float(np.mean(values))
            metrics[f'std_{key}'] = float(np.std(values))
        
        # Métricas F1
        f1_metrics = {}
        for f_key in f1_keys:
            values = [rep['f1_scores'][f_key] for rep in metrics['repetitions']]
            f1_metrics[f'avg_{f_key}'] = float(np.mean(values))
            f1_metrics[f'std_{f_key}'] = float(np.std(values))
        metrics['f1_metrics'] = f1_metrics
        
        # Matriz de confusión promedio
        conf_matrices = [rep['confusion_matrix'] for rep in metrics['repetitions']]
        metrics['avg_confusion_matrix'] = np.mean(conf_matrices, axis=0).tolist()
    
    def run(self):
        """Ejecuta el experimento para todos los modelos."""
        for model in self.models:
            try:
                result = self.run_single_model(model)
                self.results.append(result)
            except Exception as e:
                print(f"Error con modelo {model['name']}: {str(e)}")
        
        self.save_results('zero_shot_results.json')
    
    def save_results(self, filename: str):
        """Guarda los resultados en un archivo JSON."""
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(self.results, f, indent=4, ensure_ascii=False)

def main():
    # Configuración del experimento
    DATA_PATHS = (
        '../dataset/Первый_жанр_исходная.txt',
        '../dataset/Второй_жанр_исходная.txt'
    )
    
    MODELS = [
#         {'model': 'gpt2', 'name': 'GPT-2'},
#         {'model': 'facebook/opt-125m', 'name': 'OPT-125M'},#no clasifica nada en ruso
        {'model': 'sberbank-ai/rugpt3small_based_on_gpt2', 'name': 'Rugpt3small'}
    ]
    
    # Ejecutar experimento
    runner = ExperimentRunner(DATA_PATHS, MODELS)
    runner.run()

if __name__ == "__main__":
    main()

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Configuración de estilos para los gráficos
plt.style.use('seaborn')
sns.set_palette("husl")

def load_results(filename: str = 'zero_shot_results.json') -> list:
    """Carga los resultados desde el archivo JSON."""
    with open(filename, 'r', encoding='utf-8') as f:
        return json.load(f)

def plot_metrics(results: list):
    """Genera gráficos para las métricas principales."""
    # Extraer datos en formato DataFrame
    metrics_data = []
    for model in results:
        model_name = model['model_name']
        metrics = {
            'Modelo': model_name,
            'Accuracy': model['avg_accuracy'],
            'F1 Macro': model['f1_metrics']['avg_macro'],
            'ROC AUC': model['avg_roc_auc'],
            'PR AUC': model['avg_pr_auc']
        }
        metrics_data.append(metrics)
    
    df = pd.DataFrame(metrics_data).set_index('Modelo')
    
    # Gráfico de barras para métricas principales
    fig, ax = plt.subplots(figsize=(12, 6))
    df.plot(kind='bar', ax=ax)
    ax.set_title('Comparación de Métricas por Modelo', fontsize=14)
    ax.set_ylabel('Puntuación', fontsize=12)
    ax.set_xlabel('Modelo', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.6)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('metricas_principales.png', dpi=300)
    plt.show()
    
    # Gráfico de matriz de confusión promedio para cada modelo
    for model in results:
        fig, ax = plt.subplots(figsize=(6, 5))
        conf_matrix = np.array(model['avg_confusion_matrix'])
        sns.heatmap(conf_matrix, annot=True, fmt='.1f', cmap='Blues', 
                    xticklabels=['Definición', 'Descripción'],
                    yticklabels=['Definición', 'Descripción'])
        ax.set_title(f'Matriz de Confusión - {model["model_name"]}')
        ax.set_xlabel('Predicción')
        ax.set_ylabel('Verdadero')
        plt.tight_layout()
        plt.savefig(f'matriz_confusion_{model["model_name"]}.png', dpi=300)
        plt.show()

def plot_f1_scores(results: list):
    """Gráfico detallado de las métricas F1."""
    f1_data = []
    for model in results:
        model_name = model['model_name']
        f1_data.append({
            'Modelo': model_name,
            'F1 Macro': model['f1_metrics']['avg_macro'],
            'F1 Ponderado': model['f1_metrics']['avg_weighted'],
            'F1 Definición': model['f1_metrics']['avg_class_0'],
            'F1 Descripción': model['f1_metrics']['avg_class_1']
        })
    
    df = pd.DataFrame(f1_data).set_index('Modelo')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    df.plot(kind='bar', ax=ax)
    ax.set_title('Desglose de Métricas F1', fontsize=14)
    ax.set_ylabel('Puntuación F1', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.6)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('f1_scores.png', dpi=300)
    plt.show()

def plot_error_bars(results: list):
    """Gráfico con barras de error para mostrar variabilidad."""
    metrics = ['accuracy', 'roc_auc', 'pr_auc']
    model_names = [m['model_name'] for m in results]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for i, metric in enumerate(metrics):
        means = [m[f'avg_{metric}'] for m in results]
        stds = [m[f'std_{metric}'] for m in results]
        
        axes[i].bar(model_names, means, yerr=stds, capsize=5)
        axes[i].set_title(metric.upper())
        axes[i].set_ylim(0, 1)
        axes[i].grid(True, linestyle='--', alpha=0.6)
        plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    fig.suptitle('Métricas con Barras de Error (media ± desviación estándar)', y=1.05)
    plt.tight_layout()
    plt.savefig('metricas_con_error_bars.png', dpi=300)
    plt.show()

def main():
    results = load_results()
    
    print("\nResumen de Métricas:")
    for model in results:
        print(f"\nModelo: {model['model_name']}")
        print(f"Accuracy: {model['avg_accuracy']:.3f} ± {model['std_accuracy']:.3f}")
        print(f"F1 Macro: {model['f1_metrics']['avg_macro']:.3f} ± {model['f1_metrics']['std_macro']:.3f}")
        print(f"ROC AUC: {model['avg_roc_auc']:.3f} ± {model['std_roc_auc']:.3f}")
    
    plot_metrics(results)
    plot_f1_scores(results)
    plot_error_bars(results)

if __name__ == "__main__":
    main()

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Configuración con manejo de errores
model_name = "facebook/opt-125m"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    # Configuración importante para evitar overflow
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
except Exception as e:
    print(f"Error al cargar el modelo: {str(e)}")
    exit()

# 2. Ejemplos de prueba
test_cases = [
    ("Фотосинтез - это процесс преобразования света в химическую энергию", 0),
    ("Лес наполнен высокими соснами, их стволы покрыты грубой корой", 1),
    ("Квадрат - геометрическая фигура с четырьмя равными сторонами", 0),
    ("Река бурлила, неся пенящиеся воды между крутыми берегами", 1)
]

# 3. Prompt optimizado
def create_prompt(text: str) -> str:
    return f"""Определи тип текста (ответь только цифрой):
1 - определение понятия
2 - описание объекта

Текст: {text}
Тип: """

# 4. Función de generación segura
def safe_generate(text: str) -> str:
    try:
        prompt = create_prompt(text)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        
        # Configuración crítica para evitar overflow
        outputs = model.generate(
            **inputs,
            max_new_tokens=3,
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.7,
            top_k=50
        )
        
        # Decodificación segura
        response = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[-1]:], 
            skip_special_tokens=True
        ).strip()
        
        return response
    except Exception as e:
        print(f"Error en generación: {str(e)}")
        return ""

# 5. Test mejorado
def run_test():
    for text, true_label in test_cases:
        response = safe_generate(text)
        pred = -1
        
        # Análisis robusto de la respuesta
        if response:
            if "1" in response[:2]:  # Busca en los primeros caracteres
                pred = 0
            elif "2" in response[:2]:
                pred = 1
            else:
                # Búsqueda de palabras clave si no hay números
                desc_words = ["описание", "описан", "2"]
                if any(word in response.lower() for word in desc_words):
                    pred = 1
                else:
                    pred = 0  # Por defecto como definición
        
        print(f"\nТекст: {text[:50]}...")
        print(f"Истинный класс: {'Описание' if true_label else 'Определение'}")
        print(f"Ответ модели: '{response}'")
        print(f"Предсказание: {'✓' if pred == true_label else '✗'} ({'Описание' if pred else 'Определение'})")

if __name__ == "__main__":
    run_test()